# **Smoke Set**

# **Imports**

In [ ]:

from earthscape.utils.constants import SMOKE_VERSION_SPLIT_DIR, DATASET_DIR, ES_VERSION_SPLIT_DIR
from earthscape.utils.seeds import set_seed
from earthscape.data.splits import splits_create_smokeset

import os
import glob
import shutil
import pandas as pd
import geopandas as gpd

# **Train-Val-Test Splits**

In [ ]:
##################################################
# Select Smoke Splits from Train/Val/Test Splits
##################################################

# create smoke set directory structure...
if not os.path.isdir(SMOKE_VERSION_SPLIT_DIR):
    os.makedirs(SMOKE_VERSION_SPLIT_DIR)

# paths to any file that has all class labels
classes_path = glob.glob(f"{DATASET_DIR}/*areas*.csv")[0]

# set seed
seed = 111
set_seed(111)


##### select and save smoke splits...
train = splits_create_smokeset(classes_path, f"{ES_VERSION_SPLIT_DIR}/train_id.geojson", split_size=7, area_threshold=0)
train.to_file(f"{SMOKE_VERSION_SPLIT_DIR}/smoke_train_id.geojson", driver='GeoJSON', index=False)

val = splits_create_smokeset(classes_path, f"{ES_VERSION_SPLIT_DIR}/val_id.geojson", split_size=7, area_threshold=0)
val.to_file(f"{SMOKE_VERSION_SPLIT_DIR}/smoke_val_id.geojson", driver='GeoJSON', index=False)

test = splits_create_smokeset(classes_path, f"{ES_VERSION_SPLIT_DIR}/test_id.geojson", split_size=7, area_threshold=0)
test.to_file(f"{SMOKE_VERSION_SPLIT_DIR}/smoke_test_id.geojson", driver='GeoJSON', index=False)

cross_test = splits_create_smokeset(classes_path, f"{ES_VERSION_SPLIT_DIR}/test_cd.geojson", split_size=7, area_threshold=0)
cross_test.to_file(f"{SMOKE_VERSION_SPLIT_DIR}/smoke_test_cd.geojson", driver='GeoJSON', index=False)


## *Save Smoke Patches*

In [7]:
##################################################
# Copy and save smoke set images to data folder.
##################################################

# output directory
smoke_patches_dir = f"{SMOKE_VERSION_SPLIT_DIR}/patches"
if not os.path.isdir(smoke_patches_dir):
    os.makedirs(smoke_patches_dir)


# find directories containing GeoTIFF files...
patch_dirs = []
for current_dir, subdirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.lower().endswith('.tif'):
            patch_dirs.append(current_dir)
            break


# list of all smoke patch IDs
all_patch_ids = train['patch_id'].astype(str).to_list()\
                + val['patch_id'].to_list()\
                + test['patch_id'].astype(str).to_list()\
                + cross_test['patch_id'].astype(str).to_list()


##### get paths to patches...
paths = []
for id in all_patch_ids:
    for pd in patch_dirs:
        match_img = glob.glob(f"{pd}/{id}_*.tif")
        match_csv = glob.glob(f"{pd}/{id}_*.csv")
        if len(match_img) > 0:
            paths.extend(match_img)
            paths.extend(match_csv)


##### save data to smoke dataset directory...
for src in paths:
    dst = f"{smoke_patches_dir}/{os.path.basename(src)}"
    shutil.copy2(src, dst)
    

## *Save Global Smoke Files*

In [ ]:
# #####################################################################################
# # Save global files to mimic same structure as other local area dataset directories.
# #####################################################################################


# ##### get smoke set patch IDs...
# smoke_ids = train['patch_id'].astype(str).to_list()\
#             + val['patch_id'].to_list()\
#             + test['patch_id'].astype(str).to_list()\
#             + cross_test['patch_id'].astype(str).to_list()


# ##### extract areas, patches, and stats files for smoke set...
# area = pd.read_csv(GLOBAL_AREAS_PATH)
# area = area.loc[area['patch_id'].isin(smoke_ids)]
# area.to_csv(f"{SMOKE_DIR}/smoke_{STR_VERSION}_areas.csv", index=False)

# patches = gpd.read_file(GLOBAL_PATCHES_PATH)
# patches = patches.loc[patches['patch_id'].isin(smoke_ids)]
# patches.to_file(f"{SMOKE_DIR}/smoke_{STR_VERSION}_patches.geojson", driver='GeoJSON', index=False)

# src = f"{SPLIT_DIR}/{STR_VERSION}_train_stats.csv"
# dst = f"{SMOKE_DIR}/smoke_{STR_VERSION}_stats.csv"
# shutil.copy2(src, dst)
